In [24]:
import sys

sys.path.append("..")

import torch
import torchmetrics
from lightning import Trainer, seed_everything

from src import LightningDataset, Module
from src.constants import DEFAULT_SEED
from src.datasets import CustomDataset
from src.transforms import LineGraph

In [25]:
CKPT = "../lightning_logs/version_45927073/checkpoints/epoch=101-step=131378.ckpt"
BATCH_SIZE = 1

In [26]:
_ = seed_everything(DEFAULT_SEED, verbose=False)

In [27]:
ckpt = torch.load(CKPT, map_location="cpu", weights_only=False)
k = ckpt["datamodule_hyper_parameters"]["k"]

In [28]:
from pprint import pprint

ckpt_params = ckpt["hyper_parameters"] | ckpt["datamodule_hyper_parameters"]
pprint(ckpt_params)

{'batch_size': 256,
 'compile': True,
 'dataset': None,
 'dataset_name': 'csg',
 'force_reload': False,
 'k': 14,
 'lengths': (0.7, 0.2, 0.1),
 'lr': 0.0027542287033381664,
 'max_iters': 386700,
 'model_kwargs': {'angle_expansion_units': 128,
                  'classification_layers': 3,
                  'classification_units': 512,
                  'dropout': 0.1,
                  'edge_expansion_units': 256,
                  'n_bond_conv': 6,
                  'num_radial': 120},
 'model_name': 'cegann',
 'num_classes': 156,
 'num_workers': 8,
 'pre_filters': None,
 'pre_transforms': LineGraph(),
 'pred_dataset': None,
 'transforms': RandomPerturbation(stddev=0.1),
 'use_imbalance_sampler': True,
 'warmup': 100}


In [29]:
datamodule = LightningDataset(
    pred_dataset=CustomDataset(pre_transform=LineGraph(), k=k),
    # pred_dataset=CustomDataset(root="../data/test", pre_transform=LineGraph(), k=k),
    num_workers=4,
    batch_size=BATCH_SIZE,
    k=ckpt_params["k"],
)

Processing...


RuntimeError: No data found in data/custom/raw.

In [ ]:
metrics = torchmetrics.MetricCollection(
    {
        "f1": torchmetrics.F1Score(task="multiclass", num_classes=ckpt_params["num_classes"]),
        "auroc": torchmetrics.AUROC(task="multiclass", num_classes=ckpt_params["num_classes"]),
        "acc": torchmetrics.Accuracy(task="multiclass", num_classes=ckpt_params["num_classes"]),
        "confmat": torchmetrics.ConfusionMatrix(
            task="multiclass", num_classes=ckpt_params["num_classes"]
        ),
    }
)

In [ ]:
model = Module.load_from_checkpoint(checkpoint_path=CKPT, metrics=metrics, weights_only=False)

In [ ]:
trainer = Trainer(
    precision="16-mixed" if torch.cuda.is_available() else 32,
    deterministic=False,
    enable_progress_bar=True,
)

In [ ]:
predictions = trainer.predict(model=model, datamodule=datamodule)